# Exercise 0 - PydanticAI and OpenRouter

In this exercise, you get to work with data validation using Pydantic v2 and PydanticAI. Output in general from LLMs are unstructured and hard to work with, to get any value from it you need to structure the output and validate its contents. This can be done with Pydantic.


## 0. Summarize job ads

TODO:

In the data folder, there are a few job ads taken from arbetsförmedlingen.se.

a) Read all the jobs ads into python

b) Create a function that uses PydanticAI agent to summarize a job ad. This function should take in an ad as its parameter and return a summary.

c) Now create and export markdown files for each job ad and its corresponding summary.

d) Try to take other job ads from arbetsförmedlingen.se and see how well your function performs.

e) Make the output structured using a pydantic model together with PydanticAI agent. The output from the agent should have the following fields:

job_title
description
summary
responsibilities
words_in_article

## a) Read all the jobs ads into python

In [1]:
job_ads = []

for file_name in ["ads1.txt", "ads2.txt", "ads3.txt"]:
    with open(f"data/{file_name}", "r", encoding="utf-8") as f:
        job_ads.append(f.read())

len(job_ads), job_ads[0][:300]

(3,
 'About the team\n\nThe Data Platform team is newly formed and it will have two main roles in play, Data Engineer and Analytics Engineer. We have already developed a comprehensive way of working for ourselves and our stakeholders and built our solutions on a modern data stack using FiveTran, Python, DBT')

In [2]:
[ad[:100] for ad in job_ads]    

['About the team\n\nThe Data Platform team is newly formed and it will have two main roles in play, Data',
 'Engineering Operations Technician , Data Center Engineering Operations\nIdentifiant du poste: 3078595',
 'At NOBA Bank Group, we are unlocking new possibilities as we enter an exciting phase with great unta']

## b) Create a function that uses PydanticAI agent to summarize a job ad. This function should take in an ad as its parameter and return a summary.

In [6]:
from pydantic_ai import Agent
from constants import MODEL_LARGE

summarizer_agent = Agent(
    MODEL_LARGE,
    system_prompt="""
    You summarize job ads clearly and briefly.
    Focus on:
    - the role
    - main responsibilities
    - required skills
    - important details
    Keep the summary short.
    """
)

async def summarize_job_ad(ad):
    result = await summarizer_agent.run(
        user_prompt=f"""
        Summarize this job ad in plain text.
        Do not use markdown.
        Do not use bullet points.
        Keep it short and clear.

        Job ad:
        {ad}
        """
    )
    return result.output.replace("\\n", "\n")

In [8]:
summary_1 = await summarize_job_ad(job_ads[0])
summary_1

'The Data Platform team at Instabee is seeking a Data Engineer with an interest in Analytics Engineering to help build and maintain a modern data platform on GCP and Snowflake using FiveTran, Python, DBT, Airflow, and Tableau. Responsibilities include designing and implementing data integration patterns, migrating source integrations, setting up and maintaining infrastructure, creating CI/CD pipelines with IaC, orchestrating pipelines, modeling data with DBT and SQL, and contributing to analytical projects that support business decisions. Required experience covers building data platforms on GCP/Snowflake, integrating APIs, MongoDB, MySQL, using Airflow or similar, DBT/SQL modeling, Python (or Java/Go); Docker/Kubernetes and streaming experience are a plus. Candidates should have a technical degree, enjoy collaborating with stakeholders, communicate findings effectively, work agile, and be based in Stockholm open to hybrid work. The role offers a modern office with city views, a dog‑fr

## c) Now create and export markdown files for each job ad and its corresponding summary.

In [9]:
summaries = []

for ad in job_ads:
    summary = await summarize_job_ad(ad)
    summaries.append(summary)

In [11]:
for i in range(len(job_ads)):
    with open(f"data/job_ad_{i+1}.md", "w", encoding="utf-8") as f:
        f.write(job_ads[i])
        f.write("\n\n")
        f.write(summaries[i])

In [12]:
import os
os.listdir("data")

['ads3.txt',
 'ads2.txt',
 'job_ad_1.md',
 'ads1.txt',
 'job_ad_2.md',
 'job_ad_3.md']

## d) Try to take other job ads from arbetsförmedlingen.se and see how well your function performs.

## e) Make the output structured using a pydantic model together with PydanticAI agent. The output from the agent should have the following fields:

In [13]:
from pydantic import BaseModel

class JobAdOutput(BaseModel):
    job_title: str
    description: str
    summary: str
    responsibilities: list[str]
    words_in_article: int


In [14]:
from pydantic_ai import Agent
from constants import MODEL_LARGE

structured_summarizer_agent = Agent(
    MODEL_LARGE,
    output_type=JobAdOutput,
    system_prompt="""
    Extract information from a job ad and return structured output.
    """
)



In [18]:
async def extract_job_ad(ad):
    result = await structured_summarizer_agent.run(
        user_prompt=ad
    )
    return result.output